## Splyce: SIMD Vectorization of Sparse Coiteration

This notebook is created to run on Chameleon testbed for PACT 2026 Chameleon Reproducibility Challenge. This notebook is not considered to a reference point but rather a script to run the artifact on Chameleon testbed. Therefore always use the `README.md` file for any reference.

Please note that the absolute execution results will naturally vary depending on the underlying hardware. Because the instances available on Chameleon differ from the hardware testbed used in our paper's evaluation, you will observe variations in the raw metrics. However, these architectural differences do not negate the underlying optimizations; the relative performance gains and the overall speedup trends introduced by Splyce remain fully observable and consistent with our claims.

We have configured with a default hardware that is close to the hardware used for our evaluation. 
Site: `CHI@TACC`
Node Type: `compute_icelake_r750`
We noticed a few problems in spinning off this particular node in few scenarios and if you face the similar we provide a few alternatives.

1. 
Site: `CHI@TACC` <br>
Node Type: `compute_icelake_r650`

Don't expect to run all the experiments without any issue in the following hardwares.

2. 
Site: `CHI@UC` <br>
Node Type: `compute_cascadelake_r`

### Setting the Site and Project

In [ ]:
from chi import context, lease, server
from datetime import datetime, timedelta, timezone
from IPython.display import Image, display
import os

context.choose_site("CHI@TACC")
context.choose_project()

Use the drop-down menu above to set the site and project as needed.

### Lease Allocation

Keep in mind that the following lease is configured for 3 hours. Modify the number of hours required as per your needs.

In [ ]:
l = lease.Lease(
        name=f"{os.getenv('USER')}-splyce",
        duration=timedelta(hours=3)
)
l.add_node_reservation(amount=1, node_type="compute_icelake_r750")
l.add_fip_reservation(amount=1)
l.submit(wait_for_active=True, idempotent=True)
print(f"Lease created: {l.id}")

### Bare Metal Server Spinoff

In [ ]:
s = server.Server(
        name=f"{os.getenv('USER')}-splyce",
        reservation_id=l.node_reservations[0]["id"],
        image_name="CC-Ubuntu24.04"
)
s.submit(wait_for_active=True)

There are a few reasons the above command may fail:

1. Site chosen above might be not able to spinoff the server with the specified image. Change the site or sometimes even the node type.
2. Using the same server name multiple times will cause a crash. Therefore change the server name with some suffix: `f"{os.getenv('USER')}-splyce-v1"`

### Check the connectivity to the node via the floating IP

In [ ]:
floating_ip = l.get_reserved_floating_ips()[0]
s.associate_floating_ip(floating_ip)
s.check_connectivity(host=floating_ip)
# expect a "Connection successful" message

Any configuration failure until this point will not allow you to move further with the experiments. Try changing the sites and nodes until you get the successful network connectivity. 
If it still fails at every circumstance, create a ticket in Chameleon Help Desk for help.

### Install all dependencies

In [ ]:
s.execute("sudo apt-get update > /dev/null 2>&1 && sudo apt-get install -y cmake ninja-build build-essential python3-pip > /dev/null 2>&1 && sudo apt install -y python3-matplotlib > /dev/null 2>&1 && echo off | sudo tee /sys/devices/system/cpu/smt/control > /dev/null 2>&1")

### Clone and Build the Required LLVM Version

In [ ]:
llvm_clone = """
rm -rf llvm-project && \
git init llvm-project && \
cd llvm-project && \
git remote add origin https://github.com/llvm/llvm-project.git && \
git fetch --depth 1 origin 6a6d432550598a59605ee062bd0e35c9d452c0c5 && \
git checkout FETCH_HEAD
"""

s.execute(llvm_clone)

#### Build and Install LLVM/MLIR

In [ ]:
# ~ 5 minutes
llvm_build_cmd = """
cd llvm-project && rm -rf build && mkdir build && \
cmake -S llvm -B build -G Ninja \
    -DLLVM_ENABLE_PROJECTS="clang;clang-tools-extra;mlir;lld;openmp" \
    -DLLVM_ENABLE_RUNTIMES="all" \
    -DCMAKE_BUILD_TYPE=Release \
    -DLLVM_TARGETS_TO_BUILD="host;X86" \
    -DLLVM_INCLUDE_TESTS=OFF \
    -DLLVM_USE_LINKER=bfd \
    -DCMAKE_C_COMPILER=gcc \
    -DCMAKE_CXX_COMPILER=g++ \
    -DLLVM_ENABLE_ASSERTIONS=OFF \
    -DCMAKE_INSTALL_PREFIX=$HOME/llvm-install > /dev/null 2>&1
"""

s.execute(llvm_build_cmd)

In [ ]:
# ~15-20 minutes
s.execute("cd llvm-project && ninja -C build install > /dev/null 2>&1")

#### Support functions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def bash(command: str, output: str = "/dev/null 2>&1"):
    paths = "export LLVM_INSTALL=$HOME/llvm-install && export PATH=$LLVM_INSTALL/bin:$PATH && "
    if (output):
        full_command = paths + command + " > " + output
    else:
        full_command = paths + command
    s.execute(full_command)

def run_splyce(command: str, output: str = "/dev/null 2>&1"):
    full_command = "cd Splyce && " + command
    bash(full_command, output)

def run_experiments(command: str, output: str = ""):
    full_command = "cd experiments && " + command
    run_splyce(full_command, output)

def download_from_node(input_file: str, output_file: str):
    with s.ssh_connection() as conn:
        conn.get(input_file, output_file)

def plot_ref_gen(reference_plot: str, generated_plot: str):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # Load and plot the first image
    img1 = mpimg.imread(reference_plot)
    axes[0].imshow(img1)
    axes[0].axis("off")  # Hides the axis grid and numbers
    axes[0].set_title("Reference Plot")

    # Load and plot the second image
    img2 = mpimg.imread(generated_plot)
    axes[1].imshow(img2)
    axes[1].axis("off")
    axes[1].set_title("Generated Plot")

    # Adjusts spacing and displays them side-by-side
    plt.tight_layout()
    plt.show()

Check the LLVM installation is successful.

In [ ]:
bash("mlir-opt --version", "")

### Clone and Build Splyce

In [ ]:
bash("rm -rf Splyce && git clone https://github.com/KabilanMA/Splyce")

In [ ]:
run_splyce("cmake -S . -B build -G Ninja")
run_splyce("ninja -C build")

Check the Splyce build is successful

In [ ]:
run_splyce("./build/bin/splyce-opt --help | grep splyce", "")

### Running Example Programs

In [ ]:
run_splyce("./playground/run.sh spgemm single", "")

In [ ]:
run_splyce("./playground/run.sh spgemm multicore 32", "")

### Running Experiments

All experiments will produce output in the terminal which will reflect on the Jupyter Notebook. It can sometime overload the browser and crash the system, therefore it is supressed in the following cells but if needed, you can simply delete the second argument of the `run_experiments()` function calls.

#### Experiment 1 - Phase Ablation

TMA phase-ablation results require a node with hardware TopDown-L1 support (`PERF_METRICS` MSR), available starting with __Intel Ice Lake-SP (3rd Gen Xeon Scalable, e.g. Platinum 8380) or newer__ --- older nodes (Skylake-SP, Cascade Lake-SP, e.g. Gold 6126/6240R) lack the `topdown-retiring/topdown-bad-spec/topdown-fe-bound/topdown-be-bound events` and `perf_helper.c` will fail to initialize, silently skipping TMA collection while still reporting timing.

The only hardware we could find on ChameleonCloud to support modern TMA measurement is `compute_icelake_r750` on `CHI@TACC` site. Because of the hardware difference, the reference and generated plots might look different but in both cases, they will have lower bad speculation and higher retiring slots than the baselein.

In [ ]:
run_experiments("./run.sh phase_ablation")
# took ~40 minutes in our server

Phase Ablation experiment will produce two outputs.
1. A CSV file (`./experiments/phase_ablation/tma_results.csv`) containing the TMA numbers for the baseline and all 8 phase configurations.
2. A PNG file (`./experiments/phase_ablation/tma_breakdown_plot.png`) similar to Figure 11 from the paper.

For both outputs, we also have provided a reference execution output from our own server.

Let's print out both the reference CSV file and the current successful execution result of Phase Ablation.

Reference Output:

In [ ]:
%run experiments/phase_ablation/print_results.py experiments/phase_ablation/reference.csv

TMA table will be large, therefore to inspect the table without auto-wrapping zoom out your browser window.

Current Experiment Results:

In [ ]:
run_experiments("python3 ./phase_ablation/print_results.py ./phase_ablation/tma_results.csv")

Now Let's Download and Compare the plot.

In [ ]:
download_from_node("Splyce/experiments/phase_ablation/tma_breakdown_plot.png", "tma_breakdown_plot.png")

In [ ]:
plot_ref_gen("experiments/phase_ablation/reference.png", "tma_breakdown_plot.png")

#### Experiment 2 - Synthetic Data (Table 2)

In [ ]:
run_experiments("./run.sh table2")
# took ~40 minutes in our server

This generates a `results.csv` file inside each kernel's directory under `speedups/synthetic_data`, plus one combined `speedups/synthetic_data/speedup_summary.csv` - use `speedups/synthetic_data/speedup_summary_reference.csv` for reference.

In [ ]:
# Execution Results
print("================ Execution Results ================")
run_experiments("cat ./speedups/synthetic_data/speedup_summary.csv")

In [ ]:
# Reference Results
print("================ Reference Results ================")
!cat experiments/speedups/synthetic_data/speedup_summary_reference.csv

If you want to run just one of the five kernels, you can run it individually instead:

In [ ]:
run_experiments("./run.sh spgemm_speedup")

In [ ]:
run_experiments("./run.sh spmspv_speedup")

In [ ]:
run_experiments("./run.sh spmttkrp_speedup")

In [ ]:
run_experiments("./run.sh spmmh_speedup")

In [ ]:
run_experiments("./run.sh spttspm_speedup")

Once every kernel's `results.csv` exists, the below cell will print the combined table to the terminal and (re)write `speedup_summary.csv`.

In [ ]:
run_experiments("python3 speedups/synthetic_data/print_speedup_summary.py")

#### Experiment 3 - Vector Width Ablation (Figure 12)

In different hardware depending on their capabilities the vector width to get the optimal performance can vary. For the hardware environment we test, we found that vector width 4 seems to show the significant speedup compared to other vector width in different sparsity factors.

In [ ]:
run_experiments("./run.sh vector_width")
# took ~1.5 hours in our server

This will generate two results files:
1. `experiments/vector_width/results.csv` contains all the execution numbers - use `experiments/vector_width/reference.csv` for reference.
2. `experiments/vector_width/vector_width_speedup_plot.png` - Figure 12 - use `experiments/vector_width/reference.png` for reference.

The plot is an exact representation of the results in the CSV file. Therefore let's compare both the reference and generated plots.

In [ ]:
# download the plot
download_from_node("Splyce/experiments/vector_width/vector_width_speedup_plot.png", "vector_width_speedup_plot.png")

In [ ]:
plot_ref_gen("experiments/vector_width/reference.png", "vector_width_speedup_plot.png")

#### Experiment 4 - Sparsity Scaling (Figure 13)

This experiment is to show that Splyce's performance increase with increasing non zero density and even in extreme sparse data, Splyce does not show any significant degradation in performance.

You can simply run this experiment as follows:

In [ ]:
run_experiments("./run.sh sparsity_scaling")
# took ~30 minutes in our server

This will generate two results files:
1. `experiments/sparsity_scaling/results.csv` contains all execution numbers - use `experiments/sparsity_scaling/reference.csv` for reference
2. `experiments/sparsity_scaling/sparsity_scaling_plot.png` - Figure 13 - use `experiments/sparsity_scaling/reference.png` for reference.


The plot is an exact representation of the results in the CSV file. Therefore let's compare both the reference and generated plots.

In [ ]:
# download the plot
download_from_node("Splyce/experiments/sparsity_scaling/sparsity_scaling_plot.png", "sparsity_scaling_plot.png")

In [ ]:
plot_ref_gen("experiments/sparsity_scaling/reference.png", "sparsity_scaling_plot.png")

#### Experiment 5 - Parallel Scalability (Figure 14)

This experiment is to show the performance speedup Splyce produce for single core is almost linearly propotional to its parallel core execution. This ensure all the performance improvement are pinned to a single core and use multiple cores gives the same amount of performance as number of cores being used. There might be slight change the produced plot depending on the hardware.

You can simply run this experiment as follows:

In [ ]:
run_experiments("./run.sh multicore")
# took ~15 minutes in our server

This will generate two results files:

1. `experiments/multicore/results.csv` contains all execution numbers - use `experiments/multicore/reference.csv` for reference.
2. `experiments/multicore/speedup_plot.png` - Figure 14 - use `experiments/multicore/reference.png` for reference.

The plot is an exact representation of the results in the CSV file. Therefore let's compare both the reference and generated plots.

In [ ]:
# download the plot
download_from_node("Splyce/experiments/multicore/speedup_plot.png", "speedup_plot.png")

In [ ]:
plot_ref_gen("experiments/multicore/reference.png", "speedup_plot.png")

#### Experiment 6 - Performance on SuiteSparse Matrices (Table 3)

This is to show that Splyce not only gives performance on synthetic data but also on real-world matrices from the SuiteSparse matrix library. Matrices are not included in the repository - the script downloads each one it needs on demand (and, the first time any `_realworld` kernel runs, one-time scrapes `experiments/speedups/real_world_data/suitesparse/matrix_metadata.json` for download URLs).

You can simply run this experiment as follows:

In [ ]:
run_experiments("./run.sh realdata")
# took ~7-8 hours in our server

This downloads the required matrices, runs them on all five kernels, and produces `experiments/speedups/real_world_data/realworld_summary.csv` - compare against `experiments/speedups/real_world_data/realworld_summary_reference.csv` for reference.

In [ ]:
# Execution Results
print("================ Execution Results ================")
run_experiments("cat ./speedups/real_world_data/realworld_summary.csv")

In [ ]:
# Reference Results
print("================ Reference Results ================")
!cat experiments/speedups/real_world_data/realworld_summary_reference.csv

If you want to run just one of the five kernels, you can run it individually instead:

In [ ]:
run_experiments("./run.sh spgemm_realworld")

In [ ]:
run_experiments("./run.sh spmspv_realworld")

In [ ]:
run_experiments("./run.sh spmttkrp_realworld")

In [ ]:
run_experiments("./run.sh spmmh_realworld")

In [ ]:
run_experiments("./run.sh spttspm_realworld")

Once every kernel's `<kernel>_realworld_results.csv` exists, running the below cell print the combined table to the terminal and (re)write `realworld_summary.csv`.

In [ ]:
run_experiments("python3 speedups/real_wordl_data/print_realworld_summary.py")

#### Run all experiments with one command

In [ ]:
run_experiments("./run.sh all")
# took ~11 hours in our server
# we encourage not to run all the experiments in one go

Once all the experiments complete running, you can use the cells above in each experiments to compare the results.